In [32]:
import polars as pl
import altair as alt
from pathlib import Path
import json


In [12]:
root = Path('..')
json_path = root / 'data/jails-data/output/jails_pdfs.json'
parquet_path = root / 'data/jails-data/output/jails_pdfs.parquet'

with open(json_path, 'r') as f:
    raw_data = json.load(f)


In [ ]:
clean_data = []
for record in raw_data:
    clean_record = {}
    for key, value in record.items():
        if isinstance(value, (dict, list)):
            clean_record[key] = str(value) 
        else:
            clean_record[key] = value
    clean_data.append(clean_record)

df = pl.DataFrame(clean_data)
print(f"DataFrame has {df.height} rows and {df.width} columns")
df.write_parquet(parquet_path)
print(f"Saved to {parquet_path}")

DataFrame has 110 rows and 17 columns
Saved to ../data/jails-data/output/jails_pdfs.parquet


In [46]:
key_fields_s1 = [
    'Facility Name',  'Phone Number', 'Date', 'Address'
]
key_fields_s2 = [
    'Facility Type', 'Time of Day', 'AM or PM'
]
key_fields_s3 = [
    'Table Contents'
]


def get_stats_by_section(sec: int =1):
    
    if sec == 1:
        fields = key_fields_s1
    elif sec == 2:
        fields = key_fields_s2
    elif sec == 3:
        fields = key_fields_s3
    stats = []
    total = df.height

    for field in fields:
        # Count nulls, "N/A", "None", and empty strings
        missing_count = df.filter(
            pl.col(field).is_null() | 
            pl.col(field).eq("N/A") | 
            pl.col(field).eq("None") |
            pl.col(field).eq("")
        ).height
        
        stats.append({
            "Field": field,
            "Missing_Count": missing_count,
            "Missing_Percent": (missing_count / total) * 100,
            "Complete_Count": total - missing_count,
            "Complete_Percent": ((total - missing_count) / total) * 100
        })
        stats_df = pl.DataFrame(stats)
    return stats_df

# Convert stats to DataFrame
stats1 = get_stats_by_section(1)  
stats2 = get_stats_by_section(2)  
stats3 = get_stats_by_section(3)  
stats_df = pl.concat([
    stats1.with_columns(pl.lit("Basic Info").alias("Section")),
    stats2.with_columns(pl.lit("Secondary Info").alias("Section")),
    stats3.with_columns(pl.lit("Table Data").alias("Section"))
])

stats_df.sort("Missing_Percent", descending=True)


Field,Missing_Count,Missing_Percent,Complete_Count,Complete_Percent,Section
str,i64,f64,i64,f64,str
"""AM or PM""",50,45.454545,60,54.545455,"""Secondary Info"""
"""Facility Type""",18,16.363636,92,83.636364,"""Secondary Info"""
"""Facility Name""",3,2.727273,107,97.272727,"""Basic Info"""
"""Date""",2,1.818182,108,98.181818,"""Basic Info"""
"""Phone Number""",1,0.909091,109,99.090909,"""Basic Info"""
"""Address""",1,0.909091,109,99.090909,"""Basic Info"""
"""Time of Day""",1,0.909091,109,99.090909,"""Secondary Info"""
"""Table Contents""",0,0.0,110,100.0,"""Table Data"""


In [47]:
missing_chart = alt.Chart(stats_df).mark_bar().encode(
    x=alt.X('Missing_Percent:Q', title='% Missing'),
    y=alt.Y('Field:N', sort='-x', title='Field'),
    color=alt.value('#e45756'),  # Red color for missing
    tooltip=['Field', 'Missing_Count', alt.Tooltip('Missing_Percent:Q', format='.1f')]
).properties(
    width=600,
    title='Percentage of Missing Values by Field'
)

missing_chart

alt.Chart(...)

In [55]:
# Create a stacked bar chart showing complete vs missing
stacked_df = stats_df.unpivot(
    index=["Field"], 
    on=["Complete_Percent", "Missing_Percent"],
    variable_name="Status", 
    value_name="Percentage"
)

# Clean up status names
stacked_df = stacked_df.with_columns(
    pl.col("Status").map_elements(lambda x: "Complete" if "Complete" in x else "Missing")
)

# Create stacked bar chart
stacked_chart = alt.Chart(stacked_df).mark_bar().encode(
    x=alt.X('Percentage:Q', title='Percentage (%)'),
    y=alt.Y('Field:N', title='Field'),
    color=alt.Color('Status:N', scale=alt.Scale(
        domain=['Complete', 'Missing'],
        range=["#94adc7", "#e87e7e"]
    )),
    tooltip=['Field', 'Status', alt.Tooltip('Percentage:Q', format='.1f')]
).properties(
    width=600,
    title='Completion Rate by Field'
)

stacked_chart

/var/folders/m8/p0y07mnx0xj8nxd8l9zj7fn00000gn/T/ipykernel_7317/2304937129.py:10: MapWithoutReturnDtypeWarning: Calling `map_elements` without specifying `return_dtype` can lead to unpredictable results. Specify `return_dtype` to silence this warning.
  stacked_df = stacked_df.with_columns(


alt.Chart(...)

In [ ]:
### TODO: Which pdf has the most missing all over the sections? prioritize


In [56]:
raw_data[15]

{'Deceased Cause, Date, and Time': 'N/A',
 'Deceased on Suicide Watch': 'N/A',
 'Deceased Reporter': 'N/A',
 'Deceased Examined by Physician': 'N/A',
 'Deceased Signs of Illness': 'N/A',
 'Facility Name': '‘acility Name: Cook County Department Of Corrections\noe JVINN GC CKalifarnia Nivicinn 117 Chica,\n',
 'Facility Type': 'None',
 'Phone Number': 'Telephone #: 773.674.3474\n~ Wl CNOoONno\n',
 'Address': '\\ddress: 2700 S. California Division 11 Chicago IL 60608\n',
 'Date': 'Date of Occurrence: 3/5/2021\n',
 'Time of Day': 'Time of Occurrence: 9:15:00 AM\n',
 'AM or PM': 'None',
 'Occurrence Dictionary': {'(1452, 267)': ['Other fenarijfy\\*\n', 0.0],
  '(1450, 192)': ['Assault among Vetainees\n', 0.09375],
  '(919, 186)': ['ASSé\n', 0.11753902662993572],
  '(449, 183)': ['ex\n', 0.09185606060606061],
  '(1707, 104)': ['Seric\n', 0.09090909090909091],
  '(453, 104)': ['CsoGd\n', 0.0634765625],
  '(1446, 103)': ['rire\n', 0.09090909090909091],
  '(916, 103)': ['hovc\n', 0.1051136363636

In [19]:
df.head(5)

"Deceased Cause, Date, and Time",Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Type,Facility Name,Phone Number,Address,Date,Time of Day,AM or PM,Occurrence Dictionary,Table Contents,Injuries?,Resulting Death?,Report ID
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""None""","""Facility Name: Qian ‘ Lira ar …","""e veto TI O71~- TELE: ""","""( Address: 24 / Spree LEFF P I…","""Date of Occurrence: / ~7 on, 7…","""* 4 WNT Vw _ Time of Occurrenc…","""None""","""{'(1414, 262)': ['Other (speci…","""['Detainees Involved', 'Ree.',…","""No""","""None""","""handwritten_test_p1"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""None""","""‘ Ballity Name: Fu\eia Cosuini…","""Telephone #: 309. $Y F- ah of …","""dress; OUD Vb Kec ded nel) 1 T…","""Nee Y Dats of Occurrence: Zrfo…","""Td ated fe “ive Time of Occurr…","""None""","""{'(27, 229)': ['Figniing\n', 0…","""['-— ge re ee ee eee ee', '—',…","""No""","""None""","""Part 6_p1"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""None""","""— 0 Hy . a ERE a Sirnal Nw ""","""1! Shou 12 G/ 58 ""","""Jate of Oecurrenceal AID 4 \ T…","""i : eee ee Type of Occurranca:…","""la el tee dt tie 7 eae | athe.…","""None""","""{'(1036, 137)': ['Oc Spr\n', 0…","""['at', '=', '2.', '_—', '=', '…","""No""","""None""","""Part 6_p2"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Cot ""","""-acility Name: DuPage County J…","""____ Telephone #: (630) 407-22…","""Address: 501 N. County Farm Rd…","""Date of Occurrence: 03/02/2021…","""_ Time of Occurrence: 2000 ""","""pm. ""","""{'(24, 265)': ['Fightine\n', 0…","""['a', '—— we —__—', 'Detainees…","""No""","""None""","""Part 6_p3"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Cou ""","""Facility Name: St. Clair Count…","""__ Telephone #: 618-277-3505 ""","""Address: 700 North 5"" Street B…","""Date of Occurrence: 03/18/2021…","""_ Time of Occurrence: 1:27 ""","""a.m. ""","""{'(40, 262)': ['Fightinc\n', 0…","""['lJelainees invoiveci', 'a', …","""No""","""Yes ""","""Part 6_p4"""
